In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)

# Optional (for imbalance handling)
from sklearn.utils.class_weight import compute_class_weight


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv("/kaggle/input/q1-ka-ai-2026/Q1_data.csv")


In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
TARGET_COL = 'Delivery_Time'
if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not in df")

    plt.figure(figsize=(6,4))
    counts.plot(kind='bar')
    plt.title('Target class counts')
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    if len(counts) > 1:
        imbalance_ratio = counts.max() / counts.min()
        print('Imbalance ratio (max/min):', round(float(imbalance_ratio), 3))
else:
    plt.figure(figsize=(6,4))
    plt.hist(df[TARGET_COL].dropna(), bins=50, edgecolor='black')
    plt.title('Target distribution')
    plt.xlabel(TARGET_COL)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    print(df[TARGET_COL].describe())

In [ ]:
# Task 1: Write your code here:
df = df.copy()
df = df.drop(["Order_ID"], axis=1)

In [ ]:
# Task 2: Write your code here:

before = df.shape[0]
df = df.dropna(subset=['Weather', 'Traffic_Level','Time_of_Day',"Courier_Experience_yrs","Delivery_Time"])
print(f"Dropped {before - df.shape[0]} rows with missing target")

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

print("\nDataset Shapes")
print("X:", X.shape)
print("y:", y.shape)

# Encode Categorical Features
label_encoder = LabelEncoder()

for col in X.select_dtypes(include=["object"]).columns:
    X[col] = label_encoder.fit_transform(X[col])

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 6: Write your code here:
TARGET_COL = 'Delivery_Time'
plt.figure(figsize=(6,4))
plt.hist(df[TARGET_COL].dropna(), bins=50, edgecolor='black')
plt.title('Target distribution')
plt.xlabel(TARGET_COL)
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

print(df[TARGET_COL].describe())


In [ ]:
# Task 1: Write your code here:


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for test, remaining 80% for train
    random_state=42,      # reproducible output
    shuffle=True,         # representative splits
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:
# K-Fold Cross
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []
y_fold_predlist=[[]]
for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)
    y_fold_predlist =y_fold_pred

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
pred_test = model.predict(X_test)
plt.figure()
plt.scatter(y_test, pred_test)
plt.xlabel("True")
plt.ylabel("Predicted")
plt.title("Predicted vs True")
plt.show()

In [ ]:
# Task Bonus: Write your code here: